# Create Card Strength Input data

**Workflow:**
- Load and filter raw Scryfall data for Commander-legal cards.
- Normalize and clean card text for transformer inputs.
- Generate text embeddings using pre-trained sentence transformers.
- Engineer numerical and boolean features from card attributes.
- Export the final dataset for model training.

In [2]:
import pandas as pd
import numpy as np
import re
import ast
from typing import Dict, Any
from sentence_transformers import SentenceTransformer

from common.dataset_filter import filter_dataset
from common.text_normalizing import normalize_symbol

In [3]:
INPUT_DATA_PATH = "../data/scryfall/oracle-cards-20251022090300.json"
OUTPUT_DATA_PATH = "../data/training/card_strength_input.parquet"

In [4]:
df = pd.read_json(INPUT_DATA_PATH)
df = filter_dataset(df)

print(f"shape: {df.shape}")
df.head()

shape: (29677, 15)


,oracle_id,name,card_faces,mana_cost,cmc,type_line,oracle_text,power,toughness,loyalty,colors,color_identity,keywords,game_changer,set
0,00037840-6089-42ec-8c5c-281f9f474504,"Nissa, Worldsoul Speaker",NaN,{3}{G},4.0,Legendary Creature — Elf Druid,"Landfall — Whenever a land you control enters,...",3,3,NaN,[G],[G],[Landfall],False,drc
1,0004ebd0-dfd6-4276-b4a6-de0003e94237,Static Orb,NaN,{3},3.0,Artifact,"As long as this artifact is untapped, players ...",NaN,NaN,NaN,[],[],[],False,7ed
2,0006faf6-7a61-426c-9034-579f2cfcfa83,Sensory Deprivation,NaN,{U},1.0,Enchantment — Aura,Enchant creature\nEnchanted creature gets -3/-0.,NaN,NaN,NaN,[U],[U],[Enchant],False,m14
3,0007c283-5b7a-4c00-9ca1-b455c8dff8c3,Road of Return,NaN,{G}{G},2.0,Sorcery,Choose one —\n• Return target permanent card f...,NaN,NaN,NaN,[G],[G],[Entwine],False,c19
4,000d5588-5a4c-434e-988d-396632ade42c,Storm Crow,NaN,{1}{U},2.0,Creature — Bird,Flying (This creature can't be blocked except ...,1,2,NaN,[U],[U],[Flying],False,9ed


## Handle Multi-face Cards

- Define `prepare_transformer_input` to flatten multi-face cards into a single text string.
- Clean reminder text and normalize card name references to 'this card'.
- Apply the transformation to create a `transformer_input` column.

In [5]:
def prepare_oracle_text_for_transformer(text: str, card_name: str, remove_reminder_text: bool = False) -> str:
    if not isinstance(text, str):
        return ""

    # TODO: investigate if reminder text removal yields better results
    # remove reminder text
    if remove_reminder_text:
        text = re.sub(pattern=r"\([^)]*\)", repl="", string=text)

    # replace newlines
    text = text.replace(" \n", ". ")
    text = text.replace("\n", ". ")

    # normalize bullet points
    text = text.replace("•", " option: ")

    # normalize choice separators
    text = re.sub(pattern=r"choose\s+(one|two|one or both)\s+—", repl=r"choose \1 :", string=text, flags=re.IGNORECASE)

    # replace symbols
    text = re.sub(pattern=r"\{([^}]*)\}", repl=normalize_symbol, string=text)

    # fix spacing
    text = re.sub(pattern=r"\s+", repl=" ", string=text)
    text = re.sub(pattern=r"\.\s*\.", repl=".", string=text)
    text = text.replace(" ,", ",").replace(" .", ".").replace(" :", ":")
    text = text.lower().strip()

    # replace card name
    text = text.replace(card_name.lower(), "this card")
    return text

In [25]:
def prepare_transformer_input(row: pd.Series) -> str:
    """ Creates a structured, natural language input for the transformer. """
    
    def get_face_text(face_data: Dict[str, Any], card_name: str) -> str:
        name = face_data.get('name', card_name).lower()

        # TODO: replace mana symbols with natural language "2 generic, 1 green, 1 white mana"
        # TODO: replace x symbols with natural language "X generic mana"
        cost = face_data.get('mana_cost', 'no mana cost')
        cost = cost if cost != "" else "no mana cost"
        cmc = int(face_data.get('cmc', 0))
        type_line = face_data.get('type_line', 'unknown type').lower()
        # TODO: add rarity (common, uncommon, rare, mythic) [care for lands]
        
        # TODO: if newline + cost appears replace with "activated ability" or something
        raw_oracle = face_data.get('oracle_text', '')
        clean_oracle = prepare_oracle_text_for_transformer(text=raw_oracle, card_name=name, remove_reminder_text=False)
        
        # add stats
        # TODO: handle variable stats like X or *
        stats_str = ""
        if 'power' in face_data and pd.notna(face_data['power']):
            stats_str = f" STATS: {face_data['power']} power and {face_data['toughness']} toughness."
        elif 'loyalty' in face_data and pd.notna(face_data['loyalty']):
            stats_str = f" LOYALTY: {face_data['loyalty']}."
            
        return f"CARD: {name}. COST: {cost} ({cmc} mana). TYPE: {type_line}.{stats_str} TEXT: {clean_oracle}"

    # handle multi-face cards
    if 'card_faces' in row and isinstance(row['card_faces'], list):
        face_texts = [get_face_text(face_data=face, card_name=row['name']) for face in row['card_faces']]
        return " // ".join(face_texts)
    
    # handle normal cards
    return get_face_text(face_data=row.to_dict(), card_name=row['name'])

In [26]:
df['transformer_input'] = df.apply(prepare_transformer_input, axis=1)
print("\n".join(df.sample(20)['transformer_input'].to_list()))

CARD: root spider. COST: {3}{G} (4 mana). TYPE: creature — spider. STATS: 2 power and 2 toughness. TEXT: whenever this creature blocks, it gets +1/+0 and gains first strike until end of turn.
CARD: dragonlord kolaghan. COST: {4}{B}{R} (6 mana). TYPE: legendary creature — elder dragon. STATS: 6 power and 5 toughness. TEXT: flying, haste. other creatures you control have haste. whenever an opponent casts a creature or planeswalker spell with the same name as a card in their graveyard, that player loses 10 life.
CARD: dauthi cutthroat. COST: {1}{B} (2 mana). TYPE: creature — dauthi minion. STATS: 1 power and 1 toughness. TEXT: shadow (this creature can block or be blocked by only creatures with shadow.). 1 generic mana black mana, tap: destroy target creature with shadow.
CARD: animation module. COST: {1} (1 mana). TYPE: artifact. TEXT: whenever one or more +1/+1 counters are put on a permanent you control, you may pay 1 generic mana. if you do, create a 1/1 colorless servo artifact creat

## Generate Text Embeddings

In [27]:
def create_embeddings(df: pd.DataFrame) -> pd.DataFrame:
    model = SentenceTransformer(model_name_or_path='Alibaba-NLP/gte-modernbert-base', trust_remote_code=True)

    input_prompts = df['transformer_input'].to_list()
    print(f"Embedding {len(input_prompts)} cards...")
    embeddings = model.encode(sentences=input_prompts, batch_size=32, show_progress_bar=True)

    embed_cols = [f"embed_{i}" for i in range(embeddings.shape[1])]
    embeddings_df = pd.DataFrame(data=embeddings, columns=embed_cols, index=df.index)
    
    df_result = pd.concat(objs=[df, embeddings_df], axis=1)
    return df_result

In [28]:
embedding_df = create_embeddings(df=df)
embedding_df.head()

Embedding 29677 cards...


Batches:   0%|          | 2/928 [00:26<3:23:33, 13.19s/it]


KeyboardInterrupt: 

## Tabular Feature Engineering

- **Numerical:** Extract CMC, Power, Toughness, Loyalty, and Color count.
- **Boolean:** Flag specific card types (Planeswalker, Legendary, Saga) and attributes (X-cost, Modal, Transform).

In [ ]:
def safe_to_float(val):
    # TODO: nan is not allowed for MLP
    if pd.isna(val):
        return np.nan
    val = str(val).replace('*', '0').replace('+', '')
    try:
        return float(val)
    except ValueError:
        return np.nan
    
def count_colors(val):
    if isinstance(val, list):
        return len(val)
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return len(parsed)
        except (ValueError, SyntaxError):
            pass
    return 0

def check_is_modal(row):
    if '•' in str(row.get('oracle_text', '')):
        return 1
    faces = row.get('card_faces')
    if isinstance(faces, list) and len(faces) > 1:
        all_spells = all(('Instant' in str(face.get('type_line', '')) or 'Sorcery' in str(face.get('type_line', ''))) for face in faces)
        if all_spells:
            return 1
    return 0

def check_is_transform(row):
    faces = row.get('card_faces')
    if isinstance(faces, list) and len(faces) > 1:
        all_permanents = all(('Instant' not in str(face.get('type_line', '')) and 'Sorcery' not in str(face.get('type_line', ''))) for face in faces)
        if all_permanents:
            return 1
    return 0

# TODO: add rarity (1-4), care for Lands
df['num_cmc'] = df['cmc'].apply(safe_to_float)
df['num_power'] = df['power'].apply(safe_to_float)
df['num_toughness'] = df['toughness'].apply(safe_to_float)
df['num_loyalty'] = df['loyalty'].apply(safe_to_float)
df['num_colors'] = df['color_identity'].apply(count_colors)

# TODO: isblack, iswhite, ...
df['bool_is_game_changer'] = df['game_changer'].fillna(False).astype(int)
df['bool_power_var'] = df['power'].astype(str).str.contains(r'[\*X]', na=False).astype(int)
df['bool_toughness_var'] = df['toughness'].astype(str).str.contains(r'[\*X]', na=False).astype(int)
df['bool_is_planeswalker'] = df['type_line'].astype(str).str.contains('Planeswalker', case=False, na=False).astype(int)
df['bool_has_x_cost'] = df['mana_cost'].astype(str).str.contains('{X}', case=False, na=False).astype(int)
df['bool_is_legendary'] = df['type_line'].astype(str).str.contains('Legendary', case=False, na=False).astype(int)
df['bool_is_colorless'] = df['color_identity'].apply(lambda x: 1 if (isinstance(x, list) and len(x) == 0) or str(x) == '[]' else 0)
df['bool_is_modal'] = df.apply(check_is_modal, axis=1)
df['bool_is_transform'] = df.apply(check_is_transform, axis=1)
df['bool_is_saga'] = df['type_line'].astype(str).str.contains(r'\bSaga\b', case=False, na=False).astype(int)
df['bool_is_class'] = df['type_line'].astype(str).str.contains(r'\bClass\b', case=False, na=False).astype(int)

boolean_cols = [col for col in df.columns if col.startswith('bool_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]

## Dataset Export

- Concatenate all engineered features into a single final DataFrame.
- Save the resulting dataset as a Parquet file for high-performance retrieval during training.

In [ ]:
print("Attaching embeddings to the DataFrame...")
df_final = pd.concat([df, embedding_df], axis=1)
print(f"Final shape {df_final.shape}")
print(f"Exporting to {OUTPUT_DATA_PATH}...")
df_final.to_parquet(OUTPUT_DATA_PATH, index=False)